<a href="https://colab.research.google.com/github/miriamamin1213-ux/FinalModels/blob/main/GPT2Lora_Hancock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U torchao
import os,random,numpy as np,pandas as pd,torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,balanced_accuracy_score,f1_score,roc_auc_score
from transformers import GPT2Tokenizer,GPT2Model
from peft import LoraConfig,get_peft_model,TaskType

# Reproducibility
SEED=42
os.environ["PYTHONHASHSEED"]=str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True,warn_only=True)
torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:",device)

# Load Hancock data
import gdown,json
gdown.download(id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q",output="clinical_data.json",quiet=False)
gdown.download(id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F",output="pathological_data.json",quiet=False)
with open("clinical_data.json","r") as f: clinical=pd.DataFrame(json.load(f))
with open("pathological_data.json","r") as f: pathology=pd.DataFrame(json.load(f))
df=clinical.merge(pathology,on="patient_id",how="inner")
df=df[df["hpv_association_p16"].isin(["positive","negative"])].copy()
df["HPV"]=df["hpv_association_p16"].map({"negative":0,"positive":1})
features=["age_at_initial_diagnosis","sex","smoking_status","primarily_metastasis","first_treatment_intent","first_treatment_modality","days_to_first_treatment","adjuvant_treatment_intent","adjuvant_radiotherapy","adjuvant_radiotherapy_modality","adjuvant_systemic_therapy","adjuvant_systemic_therapy_modality","adjuvant_radiochemotherapy","primary_tumor_site","pT_stage","pN_stage","histologic_type","number_of_positive_lymph_nodes","number_of_resected_lymph_nodes","perinodal_invasion","lymphovascular_invasion_L","vascular_invasion_V","perineural_invasion_Pn","resection_status","infiltration_depth_in_mm"]

# Stratified train-test split before preprocessing
train_df,test_df=train_test_split(df,test_size=0.2,random_state=SEED,stratify=df["HPV"])
train_df=train_df.copy();test_df=test_df.copy()

# Train-only imputation
categorical_cols=[c for c in features if train_df[c].dtype==object]
numerical_cols=[c for c in features if c not in categorical_cols]
for col in categorical_cols:
    value=train_df[col].mode().iloc[0]
    train_df[col]=train_df[col].fillna(value)
    test_df[col]=test_df[col].fillna(value)
for col in numerical_cols:
    value=train_df[col].median()
    train_df[col]=train_df[col].fillna(value)
    test_df[col]=test_df[col].fillna(value)

# Clinical narrative
def patient_to_text(row):
    return (
        f"This patient was diagnosed at age {row['age_at_initial_diagnosis']}. "
        f"Sex is {row['sex']}. "
        f"Smoking status is {row['smoking_status']}. "
        f"Primary tumour site is {row['primary_tumor_site']}. "
        f"Histologic type is {row['histologic_type']}. "
        f"Pathological T stage is {row['pT_stage']}. "
        f"Pathological N stage is {row['pN_stage']}. "
        f"The patient has {row['number_of_positive_lymph_nodes']} positive lymph nodes and {row['number_of_resected_lymph_nodes']} resected lymph nodes. "
        f"Perinodal invasion status is {row['perinodal_invasion']}. "
        f"Lymphovascular invasion status is {row['lymphovascular_invasion_L']}. "
        f"Vascular invasion status is {row['vascular_invasion_V']}. "
        f"Perineural invasion status is {row['perineural_invasion_Pn']}. "
        f"Resection status is {row['resection_status']}. "
        f"Infiltration depth is {row['infiltration_depth_in_mm']} mm. "
        f"First treatment intent was {row['first_treatment_intent']}. "
        f"First treatment modality was {row['first_treatment_modality']}. "
        f"Days to first treatment was {row['days_to_first_treatment']}. "
        f"Adjuvant treatment intent was {row['adjuvant_treatment_intent']}. "
        f"Adjuvant radiotherapy was {row['adjuvant_radiotherapy']}. "
        f"Adjuvant radiotherapy modality was {row['adjuvant_radiotherapy_modality']}. "
        f"Adjuvant systemic therapy was {row['adjuvant_systemic_therapy']}. "
        f"Adjuvant systemic therapy modality was {row['adjuvant_systemic_therapy_modality']}. "
        f"Adjuvant radiochemotherapy was {row['adjuvant_radiochemotherapy']}. "
        f"Predict whether the patient is HPV positive or HPV negative."
    )

train_texts=train_df.apply(patient_to_text,axis=1).tolist()
test_texts=test_df.apply(patient_to_text,axis=1).tolist()
y_train=train_df["HPV"].astype(int).to_numpy()
y_test=test_df["HPV"].astype(int).to_numpy()
print("Dataset shape:",df.shape)
print("Train class distribution:\n",pd.Series(y_train).value_counts())
print("Test class distribution:\n",pd.Series(y_test).value_counts())

# Tokenizer
MODEL_NAME="openai-community/gpt2"
tokenizer=GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token=tokenizer.eos_token
MAX_LENGTH=128

class HPVTextDataset(Dataset):
    def __init__(self,texts,labels,tokenizer,max_length=128):
        self.texts=texts;self.labels=labels;self.tokenizer=tokenizer;self.max_length=max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self,index):
        encoding=self.tokenizer(self.texts[index],max_length=self.max_length,padding="max_length",truncation=True,return_tensors="pt")
        return {"input_ids":encoding["input_ids"].squeeze(0),"attention_mask":encoding["attention_mask"].squeeze(0),"label":torch.tensor(self.labels[index],dtype=torch.long)}

train_dataset=HPVTextDataset(train_texts,y_train,tokenizer,MAX_LENGTH)
test_dataset=HPVTextDataset(test_texts,y_test,tokenizer,MAX_LENGTH)

# DataLoaders
BATCH_SIZE=128
train_generator=torch.Generator();train_generator.manual_seed(SEED)
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,generator=train_generator,num_workers=0,pin_memory=torch.cuda.is_available())
test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())

# GPT-2 + LoRA
class HPVNetGPT2Text(nn.Module):
    def __init__(self,model_name=MODEL_NAME,num_classes=2,dropout=0.1):
        super().__init__()
        self.gpt2=GPT2Model.from_pretrained(model_name)
        self.gpt2.config.use_cache=False
        self.gpt2.config.pad_token_id=tokenizer.pad_token_id
        lora_config=LoraConfig(r=6,lora_alpha=12,target_modules=["c_attn"],lora_dropout=0.0,bias="none",task_type=TaskType.FEATURE_EXTRACTION)
        self.gpt2=get_peft_model(self.gpt2,lora_config)
        hidden_size=self.gpt2.config.hidden_size
        self.classifier=nn.Sequential(nn.LayerNorm(hidden_size),nn.Dropout(dropout),nn.Linear(hidden_size,128),nn.GELU(),nn.Dropout(dropout),nn.Linear(128,num_classes))
    def train(self,mode=True):
        super().train(mode)
        self.gpt2.eval()
        return self
    def forward(self,input_ids,attention_mask):
        h=self.gpt2(input_ids=input_ids,attention_mask=attention_mask,use_cache=False,return_dict=True).last_hidden_state
        m=attention_mask.unsqueeze(-1).expand_as(h).float()
        pooled=(h*m).sum(dim=1)/m.sum(dim=1).clamp(min=1e-9)
        return self.classifier(pooled)

model=HPVNetGPT2Text().to(device)
model.gpt2.print_trainable_parameters()

# Automatic train-only class weights
class_counts=np.bincount(y_train)
class_weights=torch.tensor(len(y_train)/(len(class_counts)*class_counts),dtype=torch.float32,device=device)
print("Class weights:",class_weights)
criterion=nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer=torch.optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),lr=1e-3,weight_decay=1e-4)

# Training
EPOCHS=500
best_test_loss=float("inf");best_test_ba=-float("inf");best_test_auc=-float("inf")
best_test_loss_epoch=0;best_test_ba_epoch=0;best_test_auc_epoch=0

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum=0.0;train_n=0
    for batch in train_loader:
        input_ids=batch["input_ids"].to(device,non_blocking=True)
        attention_mask=batch["attention_mask"].to(device,non_blocking=True)
        labels=batch["label"].to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        outputs=model(input_ids=input_ids,attention_mask=attention_mask)
        loss=criterion(outputs,labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(filter(lambda p:p.requires_grad,model.parameters()),max_norm=1.0)
        optimizer.step()
        train_loss_sum+=loss.item()*input_ids.size(0);train_n+=input_ids.size(0)
    average_train_loss=train_loss_sum/train_n

    # Test monitoring
    model.eval()
    test_loss_sum=0.0;test_n=0
    test_probabilities=[];test_predictions=[];test_targets=[]
    with torch.no_grad():
        for batch in test_loader:
            input_ids=batch["input_ids"].to(device,non_blocking=True)
            attention_mask=batch["attention_mask"].to(device,non_blocking=True)
            labels=batch["label"].to(device,non_blocking=True)
            outputs=model(input_ids=input_ids,attention_mask=attention_mask)
            test_loss=criterion(outputs,labels)
            probabilities=torch.softmax(outputs,dim=1)
            predictions=torch.argmax(probabilities,dim=1)
            test_loss_sum+=test_loss.item()*input_ids.size(0);test_n+=input_ids.size(0)
            test_probabilities.append(probabilities[:,1].cpu());test_predictions.append(predictions.cpu());test_targets.append(labels.cpu())

    average_test_loss=test_loss_sum/test_n
    epoch_probabilities=torch.cat(test_probabilities).numpy()
    epoch_predictions=torch.cat(test_predictions).numpy()
    epoch_targets=torch.cat(test_targets).numpy()
    epoch_ba=balanced_accuracy_score(epoch_targets,epoch_predictions)
    epoch_auc=roc_auc_score(epoch_targets,epoch_probabilities)

    if average_test_loss<best_test_loss:
        best_test_loss=average_test_loss;best_test_loss_epoch=epoch+1
        torch.save(model.state_dict(),"best_loss_gpt2_clinical_narrative_lora_hancock.pth")
    if epoch_ba>best_test_ba:
        best_test_ba=epoch_ba;best_test_ba_epoch=epoch+1
        torch.save(model.state_dict(),"best_ba_gpt2_clinical_narrative_lora_hancock.pth")
    if epoch_auc>best_test_auc:
        best_test_auc=epoch_auc;best_test_auc_epoch=epoch+1
        torch.save(model.state_dict(),"best_auc_gpt2_clinical_narrative_lora_hancock.pth")

    if (epoch+1)%50==0:
        print(f"Epoch {epoch+1:03d}, Train={average_train_loss:.4f}, Test={average_test_loss:.4f}, BA={epoch_ba:.4f}, AUC={epoch_auc:.4f}, Best Loss Epoch={best_test_loss_epoch}, Best BA Epoch={best_test_ba_epoch}, Best AUC Epoch={best_test_auc_epoch}")

print("\nBest Test Loss:",best_test_loss)
print("Best Test Loss Epoch:",best_test_loss_epoch)
print("Best Test BA:",best_test_ba)
print("Best Test BA Epoch:",best_test_ba_epoch)
print("Best Test AUC:",best_test_auc)
print("Best Test AUC Epoch:",best_test_auc_epoch)

# Final evaluation
def evaluate_checkpoint(checkpoint_path,checkpoint_name):
    model.load_state_dict(torch.load(checkpoint_path,map_location=device))
    model.eval()
    all_probabilities=[];all_predictions=[];all_targets=[]
    with torch.no_grad():
        for batch in test_loader:
            input_ids=batch["input_ids"].to(device,non_blocking=True)
            attention_mask=batch["attention_mask"].to(device,non_blocking=True)
            labels=batch["label"]
            outputs=model(input_ids=input_ids,attention_mask=attention_mask)
            probabilities=torch.softmax(outputs,dim=1)
            predictions=torch.argmax(probabilities,dim=1)
            all_probabilities.append(probabilities.cpu());all_predictions.append(predictions.cpu());all_targets.append(labels.cpu())
    probabilities=torch.cat(all_probabilities,dim=0).numpy()
    predicted=torch.cat(all_predictions,dim=0).numpy()
    y_true=torch.cat(all_targets,dim=0).numpy()
    results=classification_report(y_true,predicted,digits=4,zero_division=0)
    bal_acc=balanced_accuracy_score(y_true,predicted)
    f1=f1_score(y_true,predicted,zero_division=0)
    auc=roc_auc_score(y_true,probabilities[:,1])
    print(f"\n===== {checkpoint_name} =====")
    print("\nClassification report:");print(results)
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print(f"F1-score:          {f1:.4f}")
    print(f"AUC:               {auc:.4f}")

evaluate_checkpoint("best_loss_gpt2_clinical_narrative_lora_hancock.pth","Best Test Loss Checkpoint")
evaluate_checkpoint("best_ba_gpt2_clinical_narrative_lora_hancock.pth","Best Test BA Checkpoint")
evaluate_checkpoint("best_auc_gpt2_clinical_narrative_lora_hancock.pth","Best Test AUC Checkpoint")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 52.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q
To: /content/clinical_data.json
100%|██████████| 810k/810k [00:00<00:00, 3.62MB/s]
Downloading...
From: https://drive.google.com/uc?id=1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F
To: /content/pathological_data.json
100%|██████████| 458k/458k [00:00<00:00, 5.34MB/s]


Dataset shape: (332, 50)
Train class distribution:
 0    152
1    113
Name: count, dtype: int64
Test class distribution:
 0    39
1    28
Name: count, dtype: int64


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 221,184 || all params: 124,660,992 || trainable%: 0.1774
Class weights: tensor([0.8717, 1.1726], device='cuda:0')


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 050, Train=0.4524, Test=0.5267, BA=0.7875, AUC=0.8443, Best Loss Epoch=39, Best BA Epoch=34, Best AUC Epoch=46
Epoch 100, Train=0.1085, Test=1.0395, BA=0.7752, AUC=0.8535, Best Loss Epoch=39, Best BA Epoch=99, Best AUC Epoch=46
Epoch 150, Train=0.0102, Test=3.2044, BA=0.7573, AUC=0.7216, Best Loss Epoch=39, Best BA Epoch=108, Best AUC Epoch=46
Epoch 200, Train=0.0021, Test=2.9436, BA=0.7573, AUC=0.7766, Best Loss Epoch=39, Best BA Epoch=108, Best AUC Epoch=46
Epoch 250, Train=0.0001, Test=3.0917, BA=0.7981, AUC=0.8008, Best Loss Epoch=39, Best BA Epoch=108, Best AUC Epoch=46
Epoch 300, Train=0.0000, Test=3.2481, BA=0.7981, AUC=0.8013, Best Loss Epoch=39, Best BA Epoch=108, Best AUC Epoch=46
Epoch 350, Train=0.0000, Test=3.3493, BA=0.7981, AUC=0.7985, Best Loss Epoch=39, Best BA Epoch=108, Best AUC Epoch=46
Epoch 400, Train=0.0000, Test=3.4225, BA=0.7981, AUC=0.7985, Best Loss Epoch=39, Best BA Epoch=108, Best AUC Epoch=46
Epoch 450, Train=0.0000, Test=3.4766, BA=0.7981, AUC=0.799